<a href="https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page for one client.

**Tables to be used**: fact_content_daily_performance for the daily panel, and dim_content and dim_clients for joins.

**Time window**:

Decision moment: Single date. As of 2026-03-31. This is the moment the features are allowed to know about.

Feature window: This is 90 days before the decision moment. That is between 2025-12-31 and 2026-03-31. This period mirrors the 90-day trailing convention in the starter CSV.

Label window: This will be 30 days after the decision moment, meaning between 2026-04-01 and 2026-04-30. This period was selected because it is future data that is relative to the decision moment.

**Target/proxy**: a label observed from the features, that is, decline in impressions or page position over a defined future window. Trend_direction and trend_pct will not be included as part of the features.

In [3]:
import duckdb
from google.colab import userdata
con = duckdb.connect()
hf_token = userdata.get("PurpleElegantBass749671")
print(f"Token loaded: {hf_token[:6]}... (length {len(hf_token)})" if hf_token else "Token is empty/None!")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 0")

Token loaded: hf_IcB... (length 37)


┌─────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │ client_hash_id │ content_hash_id │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude │ ai_meta │ ai_other │ scroll_event

In [4]:
#Handling the gsc_data_start gap before building features
con.sql(f"""
    SELECT client_hash_id, gsc_data_start
    FROM read_parquet('{rel}/dim_clients.parquet')
    WHERE gsc_data_start <= DATE '2025-12-31'
""")

┌─────────────────────────┬────────────────┐
│     client_hash_id      │ gsc_data_start │
│         varchar         │      date      │
├─────────────────────────┼────────────────┤
│ client_0797ff3a1fc9a6a5 │ 2025-11-05     │
│ client_08a6a72ff48e62c0 │ 2025-09-24     │
│ client_08d2847f24cf89c1 │ 2025-07-21     │
│ client_0e1acc6cd57b0eba │ 2025-09-24     │
│ client_1d09b519bdde7c7a │ 2025-11-05     │
│ client_2094c6eb080311d5 │ 2025-12-17     │
│ client_23a62021009f63c4 │ 2025-09-24     │
│ client_2910fd937f0b4d9a │ 2025-09-24     │
│ client_2c32078d69f2cbad │ 2025-11-05     │
│ client_2e65897d94f60220 │ 2025-11-05     │
│            ·            │     ·          │
│            ·            │     ·          │
│            ·            │     ·          │
│ client_a60a11451483af1c │ 2025-11-16     │
│ client_b10cb2997d0c7c86 │ 2025-06-18     │
│ client_ba65e80a1116ae41 │ 2025-10-13     │
│ client_c182d11e4862a37d │ 2025-06-21     │
│ client_cd12bcfd98942aa1 │ 2025-10-20     │
│ client_d

In [5]:
con.sql(f"""
    SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 0
""")

┌─────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │ client_hash_id │ content_hash_id │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_claude │ ai_meta │ ai_other │ scroll_event

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context**

IDs and dates - report_date, client_hash_id, content_hash_id, month
Filtering flags - client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available

**Features**

gsc_clicks, gsc_sum_position, gsc_avg_position
ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec
sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, scroll_events

**Label/Proxy** - gsc_impressions

**Excluded**
ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — per-platform breakdown of sessions_ai; excluded given the 5-feature budget, and likely too sparse per content item to be reliable individually.

gsc_impressions (as a feature) — reused only in the label window; using it as both feature and label would leak the outcome into the input.


In [6]:
#If channel_sum ≈ total_sessions, that confirms the channels are exclusive buckets that together add up to the whole
# — meaning sessions_ai is real, separate traffic, not overlap

con.sql(f"""
    SELECT
        SUM(sessions_organic + sessions_direct + sessions_referral + sessions_social + sessions_paid + sessions_ai) AS channel_sum,
        SUM(ga4_sessions) AS total_sessions
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┐
│ channel_sum │ total_sessions │
│   int128    │     int128     │
├─────────────┼────────────────┤
│      825841 │        1299808 │
└─────────────┴────────────────┘

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain (or granularity) refers to the level of detail represented by a single row in a table or the output of a query result. It answers the fundamental question: What does one single row of this data represent?

In [7]:
rel = "hf://datasets/FlyRank/internship-warehouse"
month = "2026-03"

# 1. Grain check — zero rows back means the grain holds
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

In [8]:
# 2. Row count + date span
con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/*.parquet')
""")

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [9]:
# 3. Availability, using IS TRUE per the skill's warning
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month={month}/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ ga4_available_rows │
│   int64    │       int128       │
├────────────┼────────────────────┤
│    9841378 │             413966 │
└────────────┴────────────────────┘

In [10]:
feature_paths = [
    f"{rel}/fact_content_daily_performance/month=2025-12/*.parquet",
    f"{rel}/fact_content_daily_performance/month=2026-01/*.parquet",
    f"{rel}/fact_content_daily_performance/month=2026-02/*.parquet",
    f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"
]

#Feature frame cell
feature_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position_90d,
        SUM(ga4_sessions) AS sessions_90d,
        SUM(sessions_ai) AS ai_sessions_90d,
        COUNT(*) AS days_with_data
    FROM read_parquet([ {', '.join(f"'{p}'" for p in feature_paths)} ])
    WHERE report_date BETWEEN DATE '2025-12-31' AND DATE '2026-03-31'
      AND client_hash_id IN (
          SELECT client_hash_id FROM read_parquet('{rel}/dim_clients.parquet')
          WHERE gsc_data_start <= DATE '2025-12-31'
      )
    GROUP BY client_hash_id, content_hash_id
""").df()

print(feature_df.shape)
feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(288815, 7)


,client_hash_id,content_hash_id,clicks_90d,avg_position_90d,sessions_90d,ai_sessions_90d,days_with_data
0,client_3ffa76342f366962,content_436c6b9d09bc0d16,1.0,4.895445,0.0,0.0,76
1,client_3ffa76342f366962,content_97c888749a3807dd,0.0,4.875000,0.0,0.0,71
2,client_3ffa76342f366962,content_0fdebb2c172ebf88,0.0,3.000000,0.0,0.0,75
3,client_3ffa76342f366962,content_02ffc02bd9c365f8,0.0,NaN,0.0,0.0,71
4,client_3ffa76342f366962,content_53c5307e6b9d6f2b,0.0,4.333333,0.0,0.0,72


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation 1 — partial client coverage.** Only 40 of 104 clients (~38%) have GSC data starting on or before 2025-12-31, meaning they have a full 90-day feature window. Any model built on this slice systematically excludes newer clients, so findings won't generalize to clients with shorter histories.

**Limitation 2 — channel columns don't reconcile to total sessions.** Summing the six tracked channels (organic, direct, referral, social, paid, ai) for March 2026 gives 825,841 sessions, against 1,299,808 total ga4_sessions — a gap of about 36%. This means the channel breakdown is not exhaustive; a meaningful share of session volume isn't attributable to any tracked channel in this data, and any channel-level feature (including sessions_ai) should be read as a lower bound, not a complete picture.

In [11]:
#build a rough label from a label-window aggregate (April),
#join it to feature_df, get a quick baseline score,
#add a deliberately leaky column derived from the label window itself,
#watch the score jump, then remove it. Want me to draft that, or take a first pass yourself first?
# --- Step 1: pull the label-window impressions (April) ---

rel = "hf://datasets/FlyRank/internship-warehouse"

label_paths = [f"{rel}/fact_content_daily_performance/month=2026-04/*.parquet"]

label_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_label_month
    FROM read_parquet([ {', '.join(f"'{p}'" for p in label_paths)} ])
    WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
    GROUP BY client_hash_id, content_hash_id
""").df()

# trailing impressions — used ONLY to define the label, never as a model feature
trailing_impr_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_90d
    FROM read_parquet([ {', '.join(f"'{p}'" for p in feature_paths)} ])
    WHERE report_date BETWEEN DATE '2025-12-31' AND DATE '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
""").df()

# --- Step 2: build a rough label ---
trap_df = (feature_df
    .merge(trailing_impr_df, on=["client_hash_id", "content_hash_id"], how="inner")
    .merge(label_df, on=["client_hash_id", "content_hash_id"], how="inner")
)
trap_df["is_declining_label"] = (trap_df["impressions_label_month"] < trap_df["impressions_90d"]).astype(int)
print(trap_df["is_declining_label"].value_counts(normalize=True))

# --- Step 3: honest baseline, using only feature-window columns ---
from sklearn.tree import DecisionTreeClassifier
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

honest_features = ["clicks_90d", "avg_position_90d", "sessions_90d", "ai_sessions_90d"]
X_honest = trap_df[honest_features].fillna(0)
y = trap_df["is_declining_label"].values

honest_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
honest_tree.fit(X_honest, y)
honest_scores = honest_tree.predict_proba(X_honest)[:, 1]
print(f"HONEST Precision@50: {precision_at_k(honest_scores, y, 50):.3f}")

# --- Step 4: spring the trap — add a column derived straight from the label ---
trap_df["pct_change_leak"] = (
    (trap_df["impressions_label_month"] - trap_df["impressions_90d"])
    / trap_df["impressions_90d"].replace(0, np.nan)
).fillna(0)

leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
X_leaky = trap_df[honest_features + ["pct_change_leak"]].fillna(0)
leaky_tree.fit(X_leaky, y)
leaky_scores = leaky_tree.predict_proba(X_leaky)[:, 1]
print(f"LEAKY Precision@50 (with pct_change_leak): {precision_at_k(leaky_scores, y, 50):.3f}")

# --- Step 5: delete the leak, keep the honest number ---
print(f"\nFinal honest Precision@50 (no leak): {precision_at_k(honest_scores, y, 50):.3f}")
print("pct_change_leak was built directly from impressions_label_month — the same field the label")
print("comes from — so it isn't a real feature. Dropped. The honest number above is what goes in the write-up.")






FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_declining_label
1    0.503028
0    0.496972
Name: proportion, dtype: float64
HONEST Precision@50: 1.000
LEAKY Precision@50 (with pct_change_leak): 1.000

Final honest Precision@50 (no leak): 1.000
pct_change_leak was built directly from impressions_label_month — the same field the label
comes from — so it isn't a real feature. Dropped. The honest number above is what goes in the write-up.


In [12]:
# --- Client-holdout split, same pattern as notebook 02 Option 3 ---
rng = np.random.default_rng(42)
unique_clients = trap_df["client_hash_id"].unique()
shuffled = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:test_client_count])
test_mask = trap_df["client_hash_id"].isin(test_clients)

train_df, test_df = trap_df[~test_mask], trap_df[test_mask]
y_train, y_test = train_df["is_declining_label"].values, test_df["is_declining_label"].values

# Honest, out-of-sample
honest_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
honest_tree.fit(train_df[honest_features].fillna(0), y_train)
honest_test_scores = honest_tree.predict_proba(test_df[honest_features].fillna(0))[:, 1]
print(f"HONEST holdout Precision@50: {precision_at_k(honest_test_scores, y_test, 50):.3f}")

# Leaky, out-of-sample
leaky_features = honest_features + ["pct_change_leak"]
leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
leaky_tree.fit(train_df[leaky_features].fillna(0), y_train)
leaky_test_scores = leaky_tree.predict_proba(test_df[leaky_features].fillna(0))[:, 1]
print(f"LEAKY holdout Precision@50: {precision_at_k(leaky_test_scores, y_test, 50):.3f}")

HONEST holdout Precision@50: 0.760
LEAKY holdout Precision@50: 1.000


**The Trap**

The output for the first attempt demonstrated 1.000 Precision@50 for both the honest and leaky feature sets. This was not a good sign. A perfect score is evidence of overfitting, especially because a shallow tree could find clusters of 50+ rows that fit perfectly for the 288,815 rows scored in-sample.

Switching to the client-holdout split from notebook 02 (train on 80% of clients, score on the 20% held out entirely) fixed the issue because it forced the model to prove it generalizes to clients it has never seen.

**With the holdout split in place:**

Honest model (with only features from the trailing 90-day window) resulted in Precision@50 = 0.760. This is a more realistic and a genuine measure of whether the pattern generalizes to new clients.
Leaky model (same features plus pct_change_leak, a column computed directly from April's label-window impressions) resulted in Precision@50 = 1.000 even on clients the model never trained on. The leak alone was enough to reconstruct the label almost perfectly, because pct_change_leak is arithmetically derived from the same field the label comes from.

The leak still worked because a leaked feature does not require memorization because it is matehmatically tied to the label regardless of the client it is scored on. Leakage can be more dangerous than overfitting. Using a client-holdout split can address overfitting, but is not the best way to catch leakage because the leak is part of the feature.

The best way to address this is to check, feature by feature, whether it could have existed before the decision moment.

Kept: the honest number, 0.760. pct_change_leak is deleted from the final feature set.




## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.